In [1]:
import sys, time, contextlib, io, re, json
sys.path.append("..")

import numpy as np
import igraph as ig
import pandas as pd
import matplotlib.pyplot as plt
from torch_geometric.datasets import Planetoid, CitationFull
from sklearn.metrics import adjusted_mutual_info_score
from infomap import Infomap
from pathlib import Path

import src.optimize as opt
import src.map_equation as meq
from src.utils import compare_partitions

# load cora, coraml datasets

In [2]:
# Cora
dataset_cora = Planetoid(root="../data/Cora", name="Cora")
data_cora = dataset_cora[0]
edges_cora = list(zip(data_cora.edge_index[0].tolist(), data_cora.edge_index[1].tolist()))
g_cora_full = ig.Graph(n=data_cora.num_nodes, edges=edges_cora, directed=not data_cora.is_undirected())
if data_cora.is_undirected():
    g_cora_full = g_cora_full.simplify()

components = g_cora_full.connected_components()
giant_idx = max(components, key=len)
g_cora = g_cora_full.induced_subgraph(giant_idx)
y_true_cora = data_cora.y.numpy()[giant_idx]
print(f"Cora (gcc): N={g_cora.vcount()}, E={g_cora.ecount()}, connected={g_cora.is_connected()}")

# CoraML
dataset_coraml = CitationFull(root="../data/CoraML", name="Cora_ML", to_undirected=False)
data_coraml = dataset_coraml[0]
edges_coraml = list(zip(data_coraml.edge_index[0].tolist(), data_coraml.edge_index[1].tolist()))
g_coraml_full = ig.Graph(n=data_coraml.num_nodes, edges=edges_coraml, directed=not data_coraml.is_undirected())
if data_coraml.is_undirected():
    g_coraml_full = g_coraml_full.simplify()

components_ml = g_coraml_full.connected_components(mode="weak")
giant_idx_ml = max(components_ml, key=len)
g_coraml = g_coraml_full.induced_subgraph(giant_idx_ml)
y_true_coraml = data_coraml.y.numpy()[giant_idx_ml]
print(f"Cora_ML (gcc): N={g_coraml.vcount()}, E={g_coraml.ecount()}, connected(weak)={g_coraml.is_connected(mode='weak')}")

Cora (gcc): N=2485, E=5069, connected=True
Cora_ML (gcc): N=2810, E=8229, connected(weak)=True


# custom

In [3]:
# Cora
np.random.seed(42)
t0 = time.time()
comms_custom_cora = opt.search_community_partition(g_cora, num_restarts=10, teleportation="uniform", verbose=False)
elapsed_custom_cora = time.time() - t0
L_custom_cora = meq.compute_description_length(g_cora, comms_custom_cora)
print(f"Cora — custom: {elapsed_custom_cora:.1f}s, L={L_custom_cora:.4f} bits, {len(np.unique(comms_custom_cora))} communities")

# CoraML
np.random.seed(42)
t0 = time.time()
comms_custom_coraml = opt.search_community_partition(g_coraml, num_restarts=10, teleportation="uniform", verbose=False)
elapsed_custom_coraml = time.time() - t0
L_custom_coraml = meq.compute_description_length(g_coraml, comms_custom_coraml)
print(f"Cora_ML — custom: {elapsed_custom_coraml:.1f}s, L={L_custom_coraml:.4f} bits, {len(np.unique(comms_custom_coraml))} communities")

Cora — custom: 53.0s, L=6.5299 bits, 207 communities
Cora_ML — custom: 108.6s, L=6.2778 bits, 277 communities


# igraph

In [4]:
t0 = time.time()
comms_ig_result_cora = g_cora.community_infomap()
elapsed_igraph_cora = time.time() - t0
comms_igraph_cora = comms_ig_result_cora.membership
L_igraph_cora = meq.compute_description_length(g_cora, comms_igraph_cora)
print(f"Cora — igraph: {elapsed_igraph_cora:.1f}s, L={L_igraph_cora:.4f} bits, {len(np.unique(comms_igraph_cora))} communities")
print(f"  igraph's own codelength: {comms_ig_result_cora.codelength:.4f} bits")

t0 = time.time()
comms_ig_result_coraml = g_coraml.community_infomap()
elapsed_igraph_coraml = time.time() - t0
comms_igraph_coraml = comms_ig_result_coraml.membership
L_igraph_coraml = meq.compute_description_length(g_coraml, comms_igraph_coraml)
print(f"Cora_ML — igraph: {elapsed_igraph_coraml:.1f}s, L={L_igraph_coraml:.4f} bits, {len(np.unique(comms_igraph_coraml))} communities")
print(f"  igraph's own codelength: {comms_ig_result_coraml.codelength:.4f} bits")

Cora — igraph: 0.5s, L=6.5382 bits, 204 communities
  igraph's own codelength: 6.5382 bits
Cora_ML — igraph: 0.5s, L=6.7656 bits, 183 communities
  igraph's own codelength: 7.2886 bits


# infomap 

## cora

In [5]:
# Cora: flat (two-level)
t0 = time.time()
im_cora_flat = Infomap(directed=g_cora.is_directed(), two_level=True, num_trials=10, seed=42, silent=True)
for e in g_cora.es:
    im_cora_flat.add_link(e.source, e.target)
im_cora_flat.run()
comms_official_cora_flat = [im_cora_flat.get_modules()[node] for node in range(g_cora.vcount())]
elapsed_official_cora_flat = time.time() - t0
L_official_cora_flat = meq.compute_description_length(g_cora, comms_official_cora_flat)
print(f"Cora — official (flat): {elapsed_official_cora_flat:.1f}s, L={L_official_cora_flat:.4f} bits, {len(np.unique(comms_official_cora_flat))} communities")
print(f"  Infomap's own codelength: {im_cora_flat.codelength:.4f} bits")

# Cora: top level (lowest depth)
t0 = time.time()
im_cora = Infomap(directed=g_cora.is_directed(), num_trials=10, seed=42, silent=True)
for e in g_cora.es:
    im_cora.add_link(e.source, e.target)
im_cora.run()
print(f"  Infomap's own codelength (whole hierarchical model): {im_cora.codelength:.4f} bits")   

comms_official_cora_top = [im_cora.get_modules(depth_level=1)[node] for node in range(g_cora.vcount())]
elapsed_official_cora_top = time.time() - t0
L_official_cora_top = meq.compute_description_length(g_cora, comms_official_cora_top)
print(f"Cora — official (top): {elapsed_official_cora_top:.1f}s, L={L_official_cora_top:.4f} bits, {len(np.unique(comms_official_cora_top))} communities")

# Cora: finest level (highest depth)
t0 = time.time()
comms_official_cora_finest = [im_cora.get_modules(depth_level=-1)[node] for node in range(g_cora.vcount())]
elapsed_official_cora_finest = time.time() - t0
L_official_cora_finest = meq.compute_description_length(g_cora, comms_official_cora_finest)
print(f"Cora — official (finest): {elapsed_official_cora_finest:.1f}s, L={L_official_cora_finest:.4f} bits, {len(np.unique(comms_official_cora_finest))} communities")

Cora — official (flat): 6.0s, L=6.5350 bits, 206 communities
  Infomap's own codelength: 6.5350 bits
  Infomap's own codelength (whole hierarchical model): 6.1908 bits
Cora — official (top): 6.4s, L=8.1710 bits, 13 communities
Cora — official (finest): 5.8s, L=6.7164 bits, 322 communities


## coraml

In [6]:
# Cora_ML: flat (two-level)
t0 = time.time()
im_coraml_flat = Infomap(directed=g_coraml.is_directed(), two_level=True, num_trials=10, seed=42, silent=True)
for e in g_coraml.es:
    im_coraml_flat.add_link(e.source, e.target)
im_coraml_flat.run()
comms_official_coraml_flat = [im_coraml_flat.get_modules()[node] for node in range(g_coraml.vcount())]
elapsed_official_coraml_flat = time.time() - t0
L_official_coraml_flat = meq.compute_description_length(g_coraml, comms_official_coraml_flat)
print(f"Cora_ML — official (flat): {elapsed_official_coraml_flat:.1f}s, L={L_official_coraml_flat:.4f} bits, {len(np.unique(comms_official_coraml_flat))} communities")
print(f"  Infomap's own codelength: {im_coraml_flat.codelength:.4f} bits")   

# Cora_ML: top level (lowest depth)
t0 = time.time()
im_coraml = Infomap(directed=g_coraml.is_directed(), num_trials=10, seed=42, silent=True)
for e in g_coraml.es:
    im_coraml.add_link(e.source, e.target)
im_coraml.run()
print(f"  Infomap's own codelength (whole hierarchical model): {im_coraml.codelength:.4f} bits")   

comms_official_coraml_top = [im_coraml.get_modules(depth_level=1)[node] for node in range(g_coraml.vcount())]
elapsed_official_coraml_top = time.time() - t0
L_official_coraml_top = meq.compute_description_length(g_coraml, comms_official_coraml_top)
print(f"Cora_ML — official (top): {elapsed_official_coraml_top:.1f}s, L={L_official_coraml_top:.4f} bits, {len(np.unique(comms_official_coraml_top))} communities")

# Cora_ML: finest level (highest depth)
t0 = time.time()
comms_official_coraml_finest = [im_coraml.get_modules(depth_level=-1)[node] for node in range(g_coraml.vcount())]
elapsed_official_coraml_finest = time.time() - t0
L_official_coraml_finest = meq.compute_description_length(g_coraml, comms_official_coraml_finest)
print(f"Cora_ML — official (finest): {elapsed_official_coraml_finest:.1f}s, L={L_official_coraml_finest:.4f} bits, {len(np.unique(comms_official_coraml_finest))} communities")

Cora_ML — official (flat): 7.7s, L=6.2747 bits, 231 communities
  Infomap's own codelength: 4.7192 bits
  Infomap's own codelength (whole hierarchical model): 4.2965 bits
Cora_ML — official (top): 8.4s, L=7.7544 bits, 23 communities
Cora_ML — official (finest): 7.6s, L=6.3387 bits, 576 communities


In [7]:
def run_infomap_comparison(g, name):
    print(f"\n=== {name} ===")
    results = {}

    # custom (best of 10 restarts)
    np.random.seed(42)
    t0 = time.time()
    comms_custom = opt.search_community_partition(g, num_restarts=10, teleportation="uniform", verbose=False)
    elapsed_custom = time.time() - t0
    L_custom = meq.compute_description_length(g, comms_custom)
    print(f"{name} — custom: {elapsed_custom:.1f}s, L={L_custom:.4f} bits, {len(np.unique(comms_custom))} communities")
    results["custom"] = {"comms": comms_custom, "L": L_custom, "time": elapsed_custom}

    # igraph
    t0 = time.time()
    ig_result = g.community_infomap()
    elapsed_igraph = time.time() - t0
    comms_igraph = ig_result.membership
    L_igraph = meq.compute_description_length(g, comms_igraph)
    print(f"{name} — igraph: {elapsed_igraph:.1f}s, L={L_igraph:.4f} bits, {len(np.unique(comms_igraph))} communities")
    print(f"  igraph's own codelength: {ig_result.codelength:.4f} bits")
    results["igraph"] = {"comms": comms_igraph, "L": L_igraph, "time": elapsed_igraph}

    # official: flat (two-level)
    t0 = time.time()
    im_flat = Infomap(directed=g.is_directed(), two_level=True, num_trials=10, seed=42, silent=True)
    for e in g.es:
        im_flat.add_link(e.source, e.target)
    im_flat.run()
    comms_flat = [im_flat.get_modules()[node] for node in range(g.vcount())]
    elapsed_flat = time.time() - t0
    L_flat = meq.compute_description_length(g, comms_flat)
    print(f"{name} — official (flat): {elapsed_flat:.1f}s, L={L_flat:.4f} bits, {len(np.unique(comms_flat))} communities")
    print(f"  Infomap's own codelength: {im_flat.codelength:.4f} bits")
    results["official_flat"] = {"comms": comms_flat, "L": L_flat, "time": elapsed_flat}

    # official: hierarchical (top + finest, from the same run)
    t0 = time.time()
    im = Infomap(directed=g.is_directed(), num_trials=10, seed=42, silent=True)
    for e in g.es:
        im.add_link(e.source, e.target)
    im.run()
    print(f"  Infomap's own codelength (whole hierarchical model): {im.codelength:.4f} bits")

    comms_top = [im.get_modules(depth_level=1)[node] for node in range(g.vcount())]
    elapsed_top = time.time() - t0
    L_top = meq.compute_description_length(g, comms_top)
    print(f"{name} — official (top): {elapsed_top:.1f}s, L={L_top:.4f} bits, {len(np.unique(comms_top))} communities")
    results["official_top"] = {"comms": comms_top, "L": L_top, "time": elapsed_top}

    t0 = time.time()
    comms_finest = [im.get_modules(depth_level=-1)[node] for node in range(g.vcount())]
    elapsed_finest = time.time() - t0
    L_finest = meq.compute_description_length(g, comms_finest)
    print(f"{name} — official (finest): {elapsed_finest:.1f}s, L={L_finest:.4f} bits, {len(np.unique(comms_finest))} communities")
    results["official_finest"] = {"comms": comms_finest, "L": L_finest, "time": elapsed_finest}

    return results


results_cora = run_infomap_comparison(g_cora, "Cora")
results_coraml = run_infomap_comparison(g_coraml, "Cora_ML")


=== Cora ===
Cora — custom: 52.7s, L=6.5299 bits, 207 communities
Cora — igraph: 0.4s, L=6.5451 bits, 209 communities
  igraph's own codelength: 6.5451 bits
Cora — official (flat): 6.0s, L=6.5350 bits, 206 communities
  Infomap's own codelength: 6.5350 bits
  Infomap's own codelength (whole hierarchical model): 6.1908 bits
Cora — official (top): 6.4s, L=8.1710 bits, 13 communities
Cora — official (finest): 5.9s, L=6.7164 bits, 322 communities

=== Cora_ML ===
Cora_ML — custom: 108.1s, L=6.2778 bits, 277 communities
Cora_ML — igraph: 0.6s, L=6.8880 bits, 179 communities
  igraph's own codelength: 7.2846 bits
Cora_ML — official (flat): 7.8s, L=6.2747 bits, 231 communities
  Infomap's own codelength: 4.7192 bits
  Infomap's own codelength (whole hierarchical model): 4.2965 bits
Cora_ML — official (top): 8.4s, L=7.7544 bits, 23 communities
Cora_ML — official (finest): 7.6s, L=6.3387 bits, 576 communities
